# Complete Research Walkthrough

Notebook ini membaca artefak asli penelitian dan tidak menjalankan training berat secara otomatis.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


## 1. Latar belakang penelitian
Prediksi konsumsi energi H1-H24 membantu memperkirakan total pemakaian energi 24 jam ke depan.

## 2. Pengenalan dataset

In [ ]:
hourly = pd.read_csv(ROOT/'data/hourly_energy_clean.csv', parse_dates=['date'])
raw = pd.read_csv(ROOT/'energydata_complete.csv', parse_dates=['date'])
print('Raw rows:', len(raw))
print('Hourly rows:', len(hourly))
hourly.head()

## 3. Eksplorasi data

In [ ]:
ax = hourly.plot(x='date', y='Appliances_kWh', figsize=(12,4), title='Konsumsi energi hourly')
ax.set_ylabel('kWh')
plt.show()

## 4. Preprocessing dan agregasi hourly
Enam pencatatan 10 menit dijumlahkan dan dikonversi dari Wh ke kWh.

In [ ]:
sample = raw.head(6)[['date','Appliances']]
print(sample)
print('Total kWh jam contoh:', sample['Appliances'].sum()/1000)

## 5. Pembagian data berdasarkan waktu

In [ ]:
from src.stage2_pipeline import split_bounds
bounds = split_bounds(len(hourly))
print(bounds)
print(hourly.loc[[0, bounds.train_end-1, bounds.train_end, bounds.val_end-1, bounds.val_end, len(hourly)-1], ['date','Appliances_kWh']])

## 6. Quantile dan 15 interval fuzzy

In [ ]:
partitions = pd.read_csv(ROOT/'results/fuzzy_partition_results.csv')
q15 = partitions[(partitions['partition']=='quantile') & (partitions['requested_intervals']==15)].iloc[0]
print(q15['boundaries'])

## 7. Weighted FTS orde 3
Order 3 memakai tiga state historis untuk memprediksi state berikutnya. Audit Stage 7.1 menunjukkan order tinggi membawa risiko pola langka/fallback.

In [ ]:
fts = pd.read_csv(ROOT/'results/stage7/stage7_fts_order_comparison.csv')
fts[['order','validation_daily_mae','Daily MAE','n_patterns','rare_patterns','fallback_steps_test']]

## 8-11. FTS awal, residual, Random Forest, dan penggabungan

In [ ]:
residual = pd.read_csv(ROOT/'results/stage7/stage7_residual_comparison.csv')
residual[residual['model'].str.contains('FTS3-RandomForest')][['model','validation_daily_mae','Daily MAE','Hourly MAE']]

## 12-13. Prediksi H1-H24 dan total 24 jam

In [ ]:
hourly_pred = pd.read_csv(ROOT/'results/stage7/stage7_hourly_predictions.csv')
model = 'Residual FTS3-RandomForest'
example_origin = hourly_pred[(hourly_pred['model']==model) & (hourly_pred['split']=='test')]['origin_index'].iloc[0]
example = hourly_pred[(hourly_pred['model']==model) & (hourly_pred['origin_index']==example_origin)].sort_values('horizon')
print('Total aktual:', example['actual'].sum())
print('Total prediksi:', example['predicted'].sum())
example[['horizon','actual','predicted']].head()

## 14. Evaluasi model

In [ ]:
ranking = pd.read_csv(ROOT/'results/stage7/stage7_final_model_ranking.csv')
ranking[['rank','model','family','algorithm','fts_order','Daily MAE','Hourly MAE']].head(12)

## 15. Visualisasi hasil prediksi

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(example['horizon'], example['actual'], marker='o', label='Aktual')
plt.plot(example['horizon'], example['predicted'], marker='o', label=model)
plt.xlabel('Horizon')
plt.ylabel('kWh')
plt.title('Aktual vs prediksi H1-H24')
plt.legend(); plt.show()

## 16. Explainable AI menggunakan SHAP
SHAP tersedia untuk XGBoost Direct pada Stage 6. SHAP untuk Random Forest residual belum tersedia, sehingga tidak ditampilkan sebagai hasil RF.

In [ ]:
shap_global = pd.read_csv(ROOT/'results/stage6/shap_global_importance.csv')
shap_global.head()

## 17. Kesimpulan dan keterbatasan
Residual FTS3-RandomForest adalah test-best eksploratif, tetapi buruk pada validation. Audit Stage 7.1 merekomendasikan Direct ML sebagai baseline utama dan hybrid sebagai eksperimen tambahan.